## Education Analytics Prediction Platform

### Feature Engineering

\Importing Necessary Libraries

In [5]:
import numpy as np
import pandas as pd
import sys
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, "../src")
from data_cleaning import clean_dataset

import warnings
warnings.filterwarnings("ignore")

df_raw = pd.read_csv(r"C:\NG\Educational-Analytics-Platform\data\raw\student_dropout_dataset.csv")
target = "Dropout"
df, _ = clean_dataset(df_raw, target=target)

print(f"Shape: {df.shape}")
df.head(3)

2026-06-30 18:10:58,993 | data_cleaning | INFO | Starting cleaning pipeline. Shape: (10000, 19)
2026-06-30 18:10:59,008 | data_cleaning | INFO | Removed 0 duplicate rows
2026-06-30 18:10:59,047 | data_cleaning | INFO | Missing value detection: 4 columns have missing values
2026-06-30 18:10:59,057 | data_cleaning | INFO | Missing treatment done: 4 filled, 0 dropped, 0 remaining
2026-06-30 18:10:59,085 | data_cleaning | INFO | Outlier detection: 6 columns have outliers
2026-06-30 18:10:59,094 | data_cleaning | INFO | Outlier treatment: capped values in 6 columns
2026-06-30 18:10:59,096 | data_cleaning | INFO | Cleaning pipeline complete. Final shape: (10000, 19)


Shape: (10000, 19)


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0


\Datetime Features

In [2]:
def engineer_datetime_features(df):
    
    df_new = df.copy()
    created_features = []

    datetime_cols = []
    for col in df_new.columns:
        if df_new[col].dtype == "datetime64[ns]":
            datetime_cols.append(col)
        elif df_new[col].dtype == "object":
            parsed = pd.to_datetime(df_new[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                datetime_cols.append(col)

    for col in datetime_cols:
        dt = pd.to_datetime(df_new[col], errors="coerce")
        df_new[f"{col}_year"] = dt.dt.year
        df_new[f"{col}_month"] = dt.dt.month
        df_new[f"{col}_quarter"] = dt.dt.quarter
        df_new[f"{col}_day"] = dt.dt.day
        df_new[f"{col}_weekday"] = dt.dt.weekday
        created_features.extend([f"{col}_year", f"{col}_month",
                                 f"{col}_quarter", f"{col}_day", f"{col}_weekday"])
        
        df_new = df_new.drop(columns=[col])

    if created_features:
        print(f"Created {len(created_features)} datetime features from {len(datetime_cols)} column(s):")
        print(f"  {created_features}")
    else:
        print("No datetime columns found. Skipping datetime feature engineering.")

    return df_new, created_features

In [3]:
df, dt_features = engineer_datetime_features(df)
print(f"Shape after: {df.shape}")

No datetime columns found. Skipping datetime feature engineering.
Shape after: (10000, 19)


\Feature Encoding

In [7]:
def encode_features(df, target=None, low_card_max=10):
   
    df_new = df.copy()
    encoding_map = {"binary": {}, "onehot": [], "frequency": {}}

    cat_cols = [c for c in df_new.select_dtypes(include="object").columns if c != target]

    for col in cat_cols:
        n_unique = df_new[col].nunique()

        if n_unique == 2:
            le = LabelEncoder()
            df_new[col] = le.fit_transform(df_new[col])
            encoding_map["binary"][col] = dict(zip(le.classes_, le.transform(le.classes_)))

        elif 3 <= n_unique <= low_card_max:
            dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=True)
            dummies = dummies.astype(int)
            df_new = pd.concat([df_new.drop(columns=[col]), dummies], axis=1)
            encoding_map["onehot"].append(col)

        else:
            freq = df_new[col].value_counts().to_dict()
            df_new[col] = df_new[col].map(freq)
            encoding_map["frequency"][col] = freq

    print(f"Binary encoded   : {list(encoding_map['binary'].keys())}")
    print(f"One-hot encoded  : {encoding_map['onehot']}")
    print(f"Frequency encoded: {list(encoding_map['frequency'].keys())}")

    return df_new, encoding_map

In [8]:
df, encoding_map = encode_features(df, target=target)
print(f"Shape after encoding: {df.shape}")
df.head(3)

Binary encoded   : []
One-hot encoded  : []
Frequency encoded: []
Shape after encoding: (10000, 26)


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,...,Semester_Year 2,Semester_Year 3,Semester_Year 4,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD
0,1,22.1,1,25000.0,1,3.36,86.1,2,20.4,1,...,0,0,0,0,0,0,0,1,0,0
1,2,20.7,1,25000.0,1,4.30,68.0,2,44.0,0,...,0,1,0,0,0,1,0,0,0,0
2,3,22.4,1,40183.0,1,4.40,70.9,0,48.9,1,...,0,0,0,0,0,0,0,0,1,0
